#### Import Library

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LinearRegression
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
# import plotly

#### Import Dataset

In [2]:
df = pd.read_csv("./googleplaystore.csv")
display(df.head())

df_review = pd.read_csv("./googleplaystore_user_reviews.csv")
display(df_review.head())

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


#### Data Understanding

In [3]:
(df.isna().sum() / df.shape[0]) * 100

App                0.000000
Category           0.000000
Rating            13.596532
Reviews            0.000000
Size               0.000000
Installs           0.000000
Type               0.009224
Price              0.000000
Content Rating     0.009224
Genres             0.000000
Last Updated       0.000000
Current Ver        0.073794
Android Ver        0.027673
dtype: float64

In [4]:
df.isna().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

#### Data Cleaning

In [5]:
df['Type'] = df['Type'].fillna("Free")

In [6]:
df['Type'].unique()

array(['Free', 'Paid', '0'], dtype=object)

In [7]:
df[df['Type'].str.contains("0")]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


In [8]:
target = ['Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']
df.loc[10472, target] = df.loc[10472, target].shift(1)
df.iloc[10472]

/tmp/ipykernel_34991/929159389.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.9' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[10472, target] = df.loc[10472, target].shift(1)


App               Life Made WI-Fi Touchscreen Photo Frame
Category                                             None
Rating                                                1.9
Reviews                                              19.0
Size                                                 3.0M
Installs                                           1,000+
Type                                                 Free
Price                                                   0
Content Rating                                   Everyone
Genres                                                NaN
Last Updated                            February 11, 2018
Current Ver                                        1.0.19
Android Ver                                    4.0 and up
Name: 10472, dtype: object

In [9]:
df.isna().sum()

App                  0
Category             1
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 0
Price                0
Content Rating       0
Genres               1
Last Updated         0
Current Ver          8
Android Ver          2
dtype: int64

In [10]:
df['Category'] = df["Category"].fillna("TOOLS")

In [11]:
round(df.isna().sum() / df.shape[0] * 100, 2)

App                0.00
Category           0.00
Rating            13.60
Reviews            0.00
Size               0.00
Installs           0.00
Type               0.00
Price              0.00
Content Rating     0.00
Genres             0.01
Last Updated       0.00
Current Ver        0.07
Android Ver        0.02
dtype: float64

In [12]:
df['Rating'].fillna("0", inplace=True)
df['Rating'] = df['Rating'].astype(float)
mean_rating = round(df['Rating'].mean(), 1)
mean_rating

df['Reviews'] = df['Reviews'].astype(int)

/tmp/ipykernel_34991/913196263.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Rating'].fillna("0", inplace=True)


In [13]:
df.loc[(df['Rating'] == 0) & (df['Reviews'] != 0), "Rating"] = mean_rating

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          10841 non-null  float64
 3   Reviews         10841 non-null  int64  
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10841 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10841 non-null  object 
 9   Genres          10840 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10839 non-null  object 
dtypes: float64(1), int64(1), object(11)
memory usage: 1.1+ MB


In [15]:
df.dropna(inplace=True)
df.isna().sum()

App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
dtype: int64

In [16]:
df['Installs'] = df['Installs'].str.strip("+")
df['Installs'] = df['Installs'].str.replace(",", "").astype(int)

df['Price'] = df['Price'].str.strip("$")
df['Price'] = df['Price'].astype(float)

In [17]:
df['Size']

0                       19M
1                       14M
2                      8.7M
3                       25M
4                      2.8M
                ...        
10836                   53M
10837                  3.6M
10838                  9.5M
10839    Varies with device
10840                   19M
Name: Size, Length: 10830, dtype: object

In [18]:
df['Content Rating'] = df['Content Rating'].replace("Unrated", np.nan)

le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])

train_df = df[df['Content Rating'].notna()]
predict_df = df[df['Content Rating'].isna()]
x_train = train_df[['Category_Encoded', 'Rating']]
y_train = train_df['Content Rating']
x_predict = predict_df[['Category_Encoded', 'Rating']]

model = RandomForestClassifier()
model.fit(x_train, y_train)

prediction = model.predict(x_predict)
print(prediction)

df.loc[df['Content Rating'].isna(), "Content Rating"] = prediction

['Everyone' 'Everyone']


In [19]:
df.isna().sum()

App                 0
Category            0
Rating              0
Reviews             0
Size                0
Installs            0
Type                0
Price               0
Content Rating      0
Genres              0
Last Updated        0
Current Ver         0
Android Ver         0
Category_Encoded    0
dtype: int64

In [20]:
df['Content Rating_Encoding'] = le.fit_transform(df['Content Rating'])

In [21]:
df['Size'].unique()

array(['19M', '14M', '8.7M', '25M', '2.8M', '5.6M', '29M', '33M', '3.1M',
       '28M', '12M', '20M', '21M', '37M', '5.5M', '17M', '39M', '31M',
       '4.2M', '7.0M', '23M', '6.0M', '6.1M', '4.6M', '9.2M', '5.2M',
       '11M', '24M', 'Varies with device', '9.4M', '15M', '10M', '1.2M',
       '26M', '8.0M', '7.9M', '56M', '57M', '35M', '54M', '201k', '3.6M',
       '5.7M', '8.6M', '2.4M', '27M', '2.7M', '2.5M', '16M', '3.4M',
       '8.9M', '3.9M', '2.9M', '38M', '32M', '5.4M', '18M', '1.1M',
       '2.2M', '4.5M', '9.8M', '52M', '9.0M', '6.7M', '30M', '2.6M',
       '7.1M', '3.7M', '22M', '7.4M', '6.4M', '3.2M', '8.2M', '9.9M',
       '4.9M', '9.5M', '5.0M', '5.9M', '13M', '73M', '6.8M', '3.5M',
       '4.0M', '2.3M', '7.2M', '2.1M', '42M', '7.3M', '9.1M', '55M',
       '23k', '6.5M', '1.5M', '7.5M', '51M', '41M', '48M', '8.5M', '46M',
       '8.3M', '4.3M', '4.7M', '3.3M', '40M', '7.8M', '8.8M', '6.6M',
       '5.1M', '61M', '66M', '79k', '8.4M', '118k', '44M', '695k', '1.6M',
     

In [22]:
def clean_size(value):
    if pd.isna(value) or value == 'Varies with device':
        return 'Varies with device'
    
    value = str(value).lower()
    if "m" in value:
        return float(value.replace('m', '')) 
    elif 'k' in value:
        return float(value.replace('k', '')) / 1000
    
    try:
        return float(value)
    except:
        return 'Varies with device'

df['Size'] = df['Size'].apply(clean_size)
df.isna().sum()

App                        0
Category                   0
Rating                     0
Reviews                    0
Size                       0
Installs                   0
Type                       0
Price                      0
Content Rating             0
Genres                     0
Last Updated               0
Current Ver                0
Android Ver                0
Category_Encoded           0
Content Rating_Encoding    0
dtype: int64

In [23]:
df_review.dropna(inplace=True)
df_review

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.000000,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.250000,0.288462
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.400000,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.000000,0.300000
5,10 Best Foods for You,Best way,Positive,1.000000,0.300000
...,...,...,...,...,...
64222,Housing-Real Estate & Property,Most ads older many agents ..not much owner po...,Positive,0.173333,0.486667
64223,Housing-Real Estate & Property,"If photos posted portal load, fit purpose. I'm...",Positive,0.225000,0.447222
64226,Housing-Real Estate & Property,"Dumb app, I wanted post property rent give opt...",Negative,-0.287500,0.250000
64227,Housing-Real Estate & Property,I property business got link SMS happy perform...,Positive,0.800000,1.000000


In [24]:
data = pd.merge(df, df_review, how='inner', on='App')
display(data.head())
data.isna().sum()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Category_Encoded,Content Rating_Encoding,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,I hate,Negative,-0.800,0.900000


App                        0
Category                   0
Rating                     0
Reviews                    0
Size                       0
Installs                   0
Type                       0
Price                      0
Content Rating             0
Genres                     0
Last Updated               0
Current Ver                0
Android Ver                0
Category_Encoded           0
Content Rating_Encoding    0
Translated_Review          0
Sentiment                  0
Sentiment_Polarity         0
Sentiment_Subjectivity     0
dtype: int64

#### EDA

In [25]:
data.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Category_Encoded,Content Rating_Encoding,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,0,1,I hate,Negative,-0.800,0.900000


In [26]:
nltk.download('punkt')
# nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/pikachen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/pikachen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [27]:
data['Translated_Review'] = data['Translated_Review'].astype(str).str.lower()
stop_words = set(stopwords.words('english'))

def tokenize_and_filter(text):
    tokens = word_tokenize(text)
    return [word for word in tokens if word.isalnum() and word not in stop_words]

data['Tokens'] = data['Translated_Review'].apply(tokenize_and_filter)
data[['Translated_Review', 'Tokens']]

,Translated_Review,Tokens
0,a kid's excessive ads. the types ads allowed a...,"[kid, excessive, ads, types, ads, allowed, app..."
1,it bad >:(,[bad]
2,like,[like]
3,i love colors inspyering,"[love, colors, inspyering]"
4,i hate,[hate]
...,...,...
72600,nice broser slow browsing speed... make 8mbps ...,"[nice, broser, slow, browsing, speed, make, 8m..."
72601,the thing i found missing simple bookmark draw...,"[thing, found, missing, simple, bookmark, draw..."
72602,great relief unwanted pop ups showing up. what...,"[great, relief, unwanted, pop, ups, showing, w..."
72603,hoped found new go-to; love firefox pc. aside ...,"[hoped, found, new, love, firefox, pc, aside, ..."


In [28]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /home/pikachen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/pikachen/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [29]:
lemmatizer = WordNetLemmatizer()

def lemmatize(token):
    return [lemmatizer.lemmatize(word) for word in token]

data['Lemmatization'] = data['Tokens'].apply(lemmatize)
data[['Translated_Review', 'Tokens', 'Lemmatization']]

,Translated_Review,Tokens,Lemmatization
0,a kid's excessive ads. the types ads allowed a...,"[kid, excessive, ads, types, ads, allowed, app...","[kid, excessive, ad, type, ad, allowed, app, l..."
1,it bad >:(,[bad],[bad]
2,like,[like],[like]
3,i love colors inspyering,"[love, colors, inspyering]","[love, color, inspyering]"
4,i hate,[hate],[hate]
...,...,...,...
72600,nice broser slow browsing speed... make 8mbps ...,"[nice, broser, slow, browsing, speed, make, 8m...","[nice, broser, slow, browsing, speed, make, 8m..."
72601,the thing i found missing simple bookmark draw...,"[thing, found, missing, simple, bookmark, draw...","[thing, found, missing, simple, bookmark, draw..."
72602,great relief unwanted pop ups showing up. what...,"[great, relief, unwanted, pop, ups, showing, w...","[great, relief, unwanted, pop, ups, showing, w..."
72603,hoped found new go-to; love firefox pc. aside ...,"[hoped, found, new, love, firefox, pc, aside, ...","[hoped, found, new, love, firefox, pc, aside, ..."


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

data['Clean Text'] = data['Lemmatization'].apply(lambda x: ' '.join(x))
bow_vector = CountVectorizer()
bow_vector_matrix = bow_vector.fit_transform(data['Clean Text'])
bow_df = pd.DataFrame(bow_vector_matrix.toarray(), columns=bow_vector.get_feature_names_out())
display(bow_df)

tf_idf = TfidfVectorizer()
tf_idf_matrix = tf_idf.fit_transform(data['Clean Text'])
tf_df = pd.DataFrame(tf_idf_matrix.toarray(), columns=tf_idf.get_feature_names_out())
display(tf_df)


,00,000,00000,00love,03jun2018,04,0db,0w,10,100,...,انتبهو,جدا,سرقه,عينك,عيني,مفيد,نصابين,ياجماعه,슬픈,어떻게
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72600,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
72601,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
72602,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
72603,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


,00,000,00000,00love,03jun2018,04,0db,0w,10,100,...,انتبهو,جدا,سرقه,عينك,عيني,مفيد,نصابين,ياجماعه,슬픈,어떻게
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
72601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
72602,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
72603,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from gen